<a href="https://colab.research.google.com/github/Subhash-2910/flyrank-ML-T1/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Subhash-2910/flyrank-ML-T1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [12]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Starter data found. You're ready.


In [13]:
import duckdb
from google.colab import userdata

con = duckdb.connect()

hf_token = userdata.get("HF_TOKEN")

con.execute(f"""
CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')
""")

rel = "hf://datasets/FlyRank/internship-warehouse"

con.execute(f"""
CREATE OR REPLACE VIEW fact_content_daily_performance AS
SELECT *
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet',
    hive_partitioning = true
)
""")

print("Warehouse connection ready.")

Warehouse connection ready.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One modeling row will represent one pseudonymized content item for one pseudonymized client at the end of March 2026. I will aggregate its daily March performance into one feature row, then use April only as a later outcome window for an initial decline/review proxy.

The decision moment is 31 March 2026. At that point, the system can rank pages for human review using information already observed during March. It cannot use April performance to make that March decision.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features

I will use five March-only features: March impressions, March clicks, March CTR, March sessions, and within-March impression trend. Each is knowable at the March 31 decision moment because it uses only information measured during March.

### Label / proxy

My initial proxy is future decline: whether April impressions are at least 20% lower than March impressions. This is a measurable future outcome for ranking pages for review; it does not prove that refreshing a page will improve it.

### Context

`client_hash_id`, `content_hash_id`, `report_date`, `client_has_gsc`, and `client_has_ga4` are context fields. I will use hashed IDs only for joining, grouping, and checking the grain—not as model features.

### Excluded

I exclude April and later metrics from features because they are unavailable at the March decision moment and would leak future information. I also exclude product scores or recommendation flags, if present.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql("""
SELECT
  COUNT(*) AS march_rows,
  COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS distinct_grain_keys,
  COUNT(*) - COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS duplicate_keys
FROM fact_content_daily_performance
WHERE month = '2026-03'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,march_rows,distinct_grain_keys,duplicate_keys
0,9841378,9841378,0


This checks my grain claim. If `duplicate_keys` is 0, the March fact slice has one row per report date, client, and content item.

In [17]:
con.sql("""
SELECT
  COUNT(*) AS row_count,
  COUNT(DISTINCT content_hash_id) AS content_items,
  MIN(report_date) AS first_date,
  MAX(report_date) AS last_date
FROM fact_content_daily_performance
WHERE month = '2026-03'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,content_items,first_date,last_date
0,9841378,331437,2026-03-01,2026-03-31


This confirms the size of my development slice and verifies that its observed dates match the March 2026 feature window.

In [18]:
con.sql("""
SELECT
  COUNT(*) AS all_march_rows,
  COUNT(*) FILTER (WHERE client_has_gsc IS TRUE) AS gsc_available_rows,
  COUNT(*) FILTER (WHERE client_has_ga4 IS TRUE) AS ga4_available_rows,
  COUNT(*) FILTER (
    WHERE client_has_gsc IS TRUE
      AND client_has_ga4 IS TRUE
  ) AS rows_with_both_sources
FROM fact_content_daily_performance
WHERE month = '2026-03'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,all_march_rows,gsc_available_rows,ga4_available_rows,rows_with_both_sources
0,9841378,9841378,6822637,6822637


I use `IS TRUE` so missing availability values are not treated as available. I will only calculate a feature when its required source is available.

For the honest experiment, I will use only March-only features. I will deliberately add one invalid column derived from the future label, such as `leak_future_decline = future_decline`. The quick score should become unrealistically high because that input reveals the answer. I will then remove the column and keep only the honest score.

This demonstrates that a high model score is not trustworthy if any input contains information that was unavailable at the decision time.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This is an unbalanced panel: clients do not all have the same Search Console and Analytics history. Availability can therefore differ across pages and clients, and the March slice may not represent every page equally.

Also, a decline in April does not mean that refreshing the page will pay off. Declines can reflect seasonality, competition, changing user intent, tracking coverage, or search-engine changes. This project provides a transparent review-priority signal, not a causal promise or automatic action.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.